In [1]:
import json
import os
import re

# Define file paths
base_path = '/root/workspace/repos/thesis/research/react-agent-engine/data/langsmith'
files = ['vqa_dataset.json', 'ood_dataset.json']
keywords = ['batang']

results = []

for filename in files:
    filepath = os.path.join(base_path, filename)
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        continue
        
    with open(filepath, 'r') as f:
        try:
            data = json.load(f)
            print(f"Loaded {len(data)} items from {filename}")
            
            for item in data:
                inputs = item.get('inputs', {})
                user_text = inputs.get('user_text', '')
                
                if user_text:
                    # Check for whole words using regex to avoid matching 'sebuah' when looking for 'buah'
                    # pattern matches 'buah' or 'batang' as whole words (case insensitive)
                    pattern = r'\b(' + '|'.join(keywords) + r')\b'
                    if re.search(pattern, user_text, re.IGNORECASE):
                        result_item = {
                            'source_file': filename,
                            'image_url': inputs.get('image_url'),
                            'metadata': item.get('metadata'),
                            'user_text': user_text
                        }
                        results.append(result_item)
                        
        except json.JSONDecodeError:
            print(f"Error decoding JSON from {filename}")

print(f"\nFound {len(results)} matching items containing 'buah' or 'batang'.\n")

for res in results:
    print("-" * 20)
    print(f"Source: {res['source_file']}")
    print(f"Image URL: {res['image_url']}")
    print(f"User Text: {res['user_text']}")
    print(f"Metadata: {json.dumps(res['metadata'], indent=2)}")

Loaded 267 items from vqa_dataset.json
Loaded 120 items from ood_dataset.json

Found 0 matching items containing 'buah' or 'batang'.



In [2]:
import json
import os

# Define file paths
base_path = '/root/workspace/repos/thesis/research/react-agent-engine/data/langsmith'
files = ['vqa_dataset.json', 'ood_dataset.json']
target_tool = 'open_vocabulary_plant_detection'

results_tool = []

for filename in files:
    filepath = os.path.join(base_path, filename)
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        continue
        
    with open(filepath, 'r') as f:
        try:
            data = json.load(f)
            print(f"Loaded {len(data)} items from {filename}")
            
            for item in data:
                outputs = item.get('outputs', {})
                reference_tool_calls = outputs.get('reference_tool_calls', [])
                
                # Check if 'open_vocabulary_plant_detection' is in the tool calls
                has_tool = False
                for tool in reference_tool_calls:
                    if tool.get('name') == target_tool:
                        has_tool = True
                        break
                
                if has_tool:
                    inputs = item.get('inputs', {})
                    result_item = {
                        'source_file': filename,
                        'image_url': inputs.get('image_url'),
                        'metadata': item.get('metadata'),
                        # 'tool_calls': reference_tool_calls 
                    }
                    results_tool.append(result_item)
                        
        except json.JSONDecodeError:
            print(f"Error decoding JSON from {filename}")

print(f"\nFound {len(results_tool)} items with tool call '{target_tool}'.\n")

for res in results_tool:
    print("-" * 20)
    print(f"Source: {res['source_file']}")
    print(f"Image URL: {res['image_url']}")
    print(f"Metadata: {json.dumps(res['metadata'], indent=2)}")

Loaded 267 items from vqa_dataset.json
Loaded 120 items from ood_dataset.json

Found 18 items with tool call 'open_vocabulary_plant_detection'.

--------------------
Source: vqa_dataset.json
Image URL: https://ttymsbsmurxtpsrvlokw.supabase.co/storage/v1/object/public/thesis-bucket/plantwild/evaluation/images/0000_apple_black_rot.jpg
Metadata: {
  "class": "apple black rot",
  "plant": "apple",
  "pathogen_type": "fungal",
  "prompt_type": "vague_symptoms",
  "filename": "0000_apple_black_rot.jpg",
  "is_plant_related": true
}
--------------------
Source: vqa_dataset.json
Image URL: https://ttymsbsmurxtpsrvlokw.supabase.co/storage/v1/object/public/thesis-bucket/plantwild/evaluation/images/0041_cabbage_alternaria_leaf_spot.jpg
Metadata: {
  "class": "cabbage alternaria leaf spot",
  "plant": "cabbage",
  "pathogen_type": "fungal",
  "prompt_type": "general_inquiry",
  "filename": "0041_cabbage_alternaria_leaf_spot.jpg",
  "is_plant_related": true
}
--------------------
Source: vqa_datase